In [13]:
import pandas as pd
import numpy as np

# Load raw dataset
df = pd.read_csv('event_logs.csv')
df.head()

,user_id,event_type,event_time,product_id,amount,col_6,col_7,col_8,col_9,col_10,...,col_41,col_42,col_43,col_44,col_45,col_46,col_47,col_48,col_49,col_50
0,U0099,checkout,2023-06-03 04:13,P010,NaN,C,13.05,NaN,B,0.81,...,NaN,B,39.66,138.0,C,NaN,902.0,A,NaN,NaN
1,U0240,wishlist_add,2023-06-03 05:08,P020,2900.63,NaN,NaN,NaN,C,NaN,...,714.0,NaN,39.97,507.0,B,NaN,632.0,NaN,38.45,890.0
2,U0374,profile_update,2023-06-05 06:22,P028,NaN,A,NaN,NaN,B,60.06,...,NaN,NaN,NaN,293.0,NaN,NaN,394.0,NaN,NaN,490.0
3,U0122,page_view,2023-06-06 03:45,P001,NaN,C,NaN,747.0,B,NaN,...,365.0,C,67.84,705.0,A,96.06,110.0,NaN,NaN,NaN
4,U0211,wishlist_add,2023-06-03 12:38,P015,1728.27,A,40.19,515.0,A,NaN,...,NaN,C,NaN,876.0,NaN,NaN,NaN,B,NaN,NaN


### Initial Cleaning – Drop & Format

In [14]:
# Keep only relevant columns based on the report
df = df[['user_id', 'event_type', 'event_time', 'product_id', 'amount']]

# Convert event_time to datetime
df['event_time'] = pd.to_datetime(df['event_time'], errors='coerce')

# Remove duplicate rows
df.drop_duplicates(inplace=True)

### Missing Value Treatment

In [15]:
# Fill 'amount' with 0 for non-purchase events (optional)
purchase_events = ['purchase', 'checkout']
if 'amount' in df.columns:
    df['amount'] = df.apply(lambda row: 0 if row['event_type'] not in purchase_events and pd.isna(row['amount']) else row['amount'], axis=1)

# Impute other missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

### Outlier Treatment – IQR Capping

In [16]:
# Fill 'amount' with 0 if missing and not a purchase/checkout event
def cap_outliers(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    return np.clip(series, Q1 - 1.5*IQR, Q3 + 1.5*IQR)

numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns
for col in numeric_cols:
    df[col] = cap_outliers(df[col])

### Datetime Feature Engineering

In [17]:
df['event_time'] = pd.to_datetime(df['event_time'])
df['event_date'] = df['event_time'].dt.date
df['hour'] = df['event_time'].dt.hour
df['day_of_week'] = df['event_time'].dt.day_name()

### Format Floats – Round Decimals

In [18]:
# Round to 2 decimals for cleaner output
df[numeric_cols] = df[numeric_cols].round(2)

### Export Final Cleaned Dataset

In [19]:
df.to_csv("final_cleaned_event_logs.csv", index=False)